# Problem 1 — RAG over PDFs with Ollama + Llama 3.2 (Colab)

Self-contained Colab version of `Problem1_RAG_PDF_Ollama_Llama3.2`. It:

1. Installs and starts a local **Ollama** server inside the Colab VM and pulls **llama3.2**.
2. Generates 3 sample PDF documents (or lets you upload your own).
3. Chunks + embeds them into a persistent **ChromaDB** collection.
4. Answers questions with retrieval-augmented generation (RAG) using the local Llama 3.2 model.

> Tip: A GPU runtime (`Runtime > Change runtime type > T4 GPU`) is optional but makes Ollama generation noticeably faster.


## 1. Install and start Ollama, pull llama3.2

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time

ollama_proc = subprocess.Popen(
    ['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
time.sleep(5)
print('Ollama server started, pid =', ollama_proc.pid)


In [ ]:
!ollama pull llama3.2


## 2. Install Python dependencies

In [ ]:
!pip install -q chromadb pypdf ollama reportlab


## 3. Write out the project source files

In [ ]:
%%writefile rag_pipeline.py
"""Core RAG pipeline: PDF loading, chunking, vector store, and generation.

Shared by `ingest.py`, `app.py` (Streamlit UI) and `cli.py`.
"""

from __future__ import annotations

import glob
import os
from dataclasses import dataclass
from pathlib import Path

import chromadb
import ollama
from pypdf import PdfReader

BASE_DIR = Path(__file__).parent
DOCS_DIR = BASE_DIR / "documents"
CHROMA_DIR = BASE_DIR / "chroma_db"
COLLECTION_NAME = "pdf_rag_collection"

# The chat/generation model. Must already be pulled: `ollama pull llama3.2`
OLLAMA_LLM_MODEL = os.environ.get("OLLAMA_LLM_MODEL", "llama3.2")

CHUNK_SIZE = 1200        # characters per chunk
CHUNK_OVERLAP = 200      # characters shared between consecutive chunks
TOP_K = 4                # number of chunks retrieved per query


@dataclass
class RetrievedChunk:
    text: str
    source: str
    chunk_index: int
    distance: float


def _chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Simple sliding-window character chunker with overlap."""
    text = " ".join(text.split())  # normalize whitespace
    if not text:
        return []

    chunks: list[str] = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks


def extract_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def get_chroma_collection():
    """Return (creating if necessary) the persistent Chroma collection.

    Uses Chroma's built-in default embedding function (ONNX MiniLM), so no
    extra embedding model needs to be pulled through Ollama.
    """
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    return client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )


def ingest_documents(docs_dir: Path = DOCS_DIR, reset: bool = True) -> int:
    """Chunk every PDF under `docs_dir` and (re)populate the Chroma collection.

    Returns the number of chunks indexed.
    """
    client = chromadb.PersistentClient(path=str(CHROMA_DIR))
    if reset:
        try:
            client.delete_collection(COLLECTION_NAME)
        except Exception:
            pass
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},
    )

    pdf_paths = sorted(glob.glob(str(docs_dir / "*.pdf")))
    if not pdf_paths:
        raise FileNotFoundError(
            f"No PDF files found in {docs_dir}. Run create_sample_pdfs.py first, "
            "or drop your own PDFs into that folder."
        )

    ids, documents, metadatas = [], [], []
    for pdf_path in pdf_paths:
        source_name = Path(pdf_path).name
        text = extract_pdf_text(Path(pdf_path))
        chunks = _chunk_text(text)
        for i, chunk in enumerate(chunks):
            ids.append(f"{source_name}::{i}")
            documents.append(chunk)
            metadatas.append({"source": source_name, "chunk_index": i})

    if documents:
        # Chroma has a per-call batch size limit; chunk the insert.
        batch_size = 128
        for start in range(0, len(documents), batch_size):
            end = start + batch_size
            collection.add(
                ids=ids[start:end],
                documents=documents[start:end],
                metadatas=metadatas[start:end],
            )

    return len(documents)


def retrieve(query: str, top_k: int = TOP_K) -> list[RetrievedChunk]:
    collection = get_chroma_collection()
    results = collection.query(query_texts=[query], n_results=top_k)

    retrieved: list[RetrievedChunk] = []
    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]
    dists = results.get("distances", [[]])[0]
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append(
            RetrievedChunk(
                text=doc,
                source=meta.get("source", "unknown"),
                chunk_index=meta.get("chunk_index", -1),
                distance=dist,
            )
        )
    return retrieved


SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions using ONLY the provided "
    "context excerpts from the user's PDF documents. If the answer is not "
    "contained in the context, say you don't know instead of guessing. "
    "Always mention which source document(s) you used."
)


def build_prompt(query: str, chunks: list[RetrievedChunk]) -> str:
    context_blocks = []
    for c in chunks:
        context_blocks.append(f"[Source: {c.source}, chunk {c.chunk_index}]\n{c.text}")
    context = "\n\n---\n\n".join(context_blocks) if context_blocks else "(no context retrieved)"

    return (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        "Answer the question using only the context above."
    )


def answer_question(query: str, top_k: int = TOP_K, model: str = OLLAMA_LLM_MODEL) -> tuple[str, list[RetrievedChunk]]:
    """Full RAG turn: retrieve relevant chunks, then ask the Ollama LLM."""
    chunks = retrieve(query, top_k=top_k)
    user_prompt = build_prompt(query, chunks)

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
    )
    answer = response["message"]["content"]
    return answer, chunks


def stream_answer(query: str, top_k: int = TOP_K, model: str = OLLAMA_LLM_MODEL):
    """Generator yielding answer text tokens as they stream from Ollama.

    Call `retrieve(query, top_k)` separately if you also need the source
    chunks (e.g. to display citations alongside the streamed answer).
    """
    chunks = retrieve(query, top_k=top_k)
    user_prompt = build_prompt(query, chunks)

    stream = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        stream=True,
    )
    for part in stream:
        yield part["message"]["content"]


In [ ]:
%%writefile create_sample_pdfs.py
"""Generate 3 sample PDF documents used to demonstrate the RAG pipeline.

Run once before `ingest.py`:

    python create_sample_pdfs.py

This creates `documents/company_handbook.pdf`, `documents/product_manual.pdf`
and `documents/it_security_policy.pdf`. Feel free to replace these with your
own real PDFs -- the ingestion script picks up every *.pdf file it finds in
the `documents/` folder.
"""

from __future__ import annotations

import textwrap
from pathlib import Path

from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer

DOCS_DIR = Path(__file__).parent / "documents"

DOCUMENTS: dict[str, tuple[str, str]] = {
    "company_handbook.pdf": (
        "Acme Corp Employee Handbook",
        """
        Welcome to Acme Corp! This handbook explains our policies.

        Working Hours: Standard working hours are 9:00 AM to 6:00 PM,
        Monday through Friday, with a one hour lunch break. Employees may
        request flexible hours through their manager.

        Leave Policy: Full-time employees accrue 18 days of paid time off
        (PTO) per year, plus 10 public holidays. Sick leave of up to 12 days
        per year does not require advance notice, but employees should
        inform their manager as soon as possible.

        Remote Work: Employees may work remotely up to 3 days per week
        after completing a 90 day probation period, subject to manager
        approval. Fully remote arrangements require VP-level sign-off.

        Code of Conduct: All employees must treat colleagues, customers,
        and partners with respect. Harassment, discrimination, and
        retaliation of any kind are strictly prohibited and will result in
        disciplinary action up to and including termination.

        Expense Reimbursement: Business expenses (travel, client meals,
        software subscriptions) are reimbursed within 15 business days of
        submitting a receipt through the Expensify portal. The approval
        limit for a direct manager is $500 per expense; anything above that
        requires Finance approval.

        Performance Reviews: Formal performance reviews happen twice a
        year, in June and December. Employees set quarterly OKRs with their
        manager and receive a written review summarizing progress against
        those goals.
        """,
    ),
    "product_manual.pdf": (
        "NovaBrew Coffee Machine - User Manual",
        """
        NovaBrew Coffee Machine Model NB-200 User Manual.

        Setup: Remove all packaging materials, place the machine on a flat
        surface, and fill the removable 1.5 liter water tank with fresh
        cold water. Do not exceed the MAX fill line.

        Brewing a Cup: Press the power button and wait for the ready light
        to turn solid blue (approximately 30 seconds). Select your brew
        size (Small 150ml, Medium 250ml, Large 350ml) using the dial, then
        press Start. The machine automatically stops when the selected
        volume is reached.

        Descaling: The NovaBrew should be descaled every 2 months, or when
        the descale indicator light flashes amber. Use the included
        descaling solution mixed with water in a 1:4 ratio, run it through
        the machine using the Descale cycle, then run two cycles of plain
        water to rinse.

        Cleaning: The drip tray and water tank are dishwasher safe on the
        top rack. Wipe the exterior with a damp cloth only; never immerse
        the base unit in water.

        Troubleshooting: If the machine will not turn on, check that it is
        plugged into a working outlet and that the water tank is properly
        seated -- the tank has a safety sensor that blocks power when it is
        missing. If brewing is unusually slow, this usually indicates
        mineral buildup and the unit should be descaled.

        Warranty: The NovaBrew NB-200 carries a 2 year limited warranty
        covering manufacturing defects. Warranty does not cover damage from
        failure to descale the unit as recommended.
        """,
    ),
    "it_security_policy.pdf": (
        "Acme Corp IT Security Policy",
        """
        Acme Corp Information Security Policy, Version 3.

        Password Requirements: All corporate accounts must use passwords of
        at least 14 characters, combining upper case, lower case, numbers,
        and symbols. Passwords must be rotated every 180 days and must not
        be reused across the last 10 passwords.

        Multi-Factor Authentication: MFA is mandatory for all access to
        email, VPN, and cloud infrastructure (AWS, GCP, Azure) accounts.
        Approved MFA methods are the Okta Verify app or a hardware security
        key; SMS-based MFA is disallowed for administrative accounts.

        Device Policy: Only company-issued or MDM-enrolled personal
        devices may access corporate email and internal systems. Devices
        must have disk encryption enabled and a screen lock timeout of 5
        minutes or less.

        Data Classification: Data is classified as Public, Internal,
        Confidential, or Restricted. Restricted data (customer PII,
        payment information) may only be stored in approved, encrypted
        systems and may never be emailed or copied to USB drives.

        Incident Reporting: Any suspected security incident -- lost
        device, phishing email, suspicious login -- must be reported to
        security@acmecorp.example within 1 hour of discovery by emailing
        the security team or filing a ticket in the #security-incidents
        channel.

        Vendor Access: Third-party vendors requiring system access must
        sign a Data Processing Agreement and be provisioned with
        time-limited accounts that expire automatically after 90 days
        unless renewed.
        """,
    ),
}


def build_pdf(path: Path, title: str, body: str) -> None:
    styles = getSampleStyleSheet()
    doc = SimpleDocTemplate(str(path), pagesize=LETTER)
    story = [Paragraph(title, styles["Title"]), Spacer(1, 0.3 * inch)]

    for raw_paragraph in textwrap.dedent(body).strip().split("\n\n"):
        clean = " ".join(line.strip() for line in raw_paragraph.splitlines()).strip()
        if clean:
            story.append(Paragraph(clean, styles["BodyText"]))
            story.append(Spacer(1, 0.15 * inch))

    doc.build(story)


def main() -> None:
    DOCS_DIR.mkdir(parents=True, exist_ok=True)
    for filename, (title, body) in DOCUMENTS.items():
        out_path = DOCS_DIR / filename
        build_pdf(out_path, title, body)
        print(f"Wrote {out_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile ingest.py
"""Ingest all PDFs in documents/ into the persistent ChromaDB collection.

Usage:
    python ingest.py            # (re)build the index from scratch
    python ingest.py --no-reset # add to the existing index instead
"""

from __future__ import annotations

import argparse

from rag_pipeline import ingest_documents


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument(
        "--no-reset",
        action="store_true",
        help="Do not wipe the existing collection before ingesting.",
    )
    args = parser.parse_args()

    n_chunks = ingest_documents(reset=not args.no_reset)
    print(f"Indexed {n_chunks} chunks into ChromaDB.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cli.py
"""Command-line chat REPL for the PDF RAG assistant (no Streamlit needed).

Usage:
    python cli.py
"""

from __future__ import annotations

from rag_pipeline import CHROMA_DIR, OLLAMA_LLM_MODEL, answer_question


def main() -> None:
    if not CHROMA_DIR.exists():
        print("No index found yet. Run `python ingest.py` first.")
        return

    print(f"PDF RAG CLI (model: {OLLAMA_LLM_MODEL}). Type 'exit' to quit.\n")
    while True:
        try:
            query = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break

        if not query:
            continue
        if query.lower() in {"exit", "quit"}:
            break

        answer, chunks = answer_question(query)
        print(f"\nAssistant: {answer}\n")
        print("Sources:")
        for c in chunks:
            print(f"  - {c.source} (chunk {c.chunk_index}, distance={c.distance:.3f})")
        print()


if __name__ == "__main__":
    main()


## 4. Generate sample PDFs (or upload your own)

Skip the next cell and instead run:
```python
from google.colab import files
files.upload()  # upload your own .pdf files into documents/
```
if you'd rather use your own PDFs.

In [ ]:
!python create_sample_pdfs.py


## 5. Build the ChromaDB index

In [ ]:
!python ingest.py


## 6. Ask a question (RAG with Llama 3.2 via Ollama)

In [ ]:
import importlib
import rag_pipeline as rp
importlib.reload(rp)

answer, chunks = rp.answer_question(
    'How many days of remote work are employees allowed per week?'
)
print('Answer:\n', answer)
print('\nSources:')
for c in chunks:
    print(f' - {c.source} (chunk {c.chunk_index}, distance={c.distance:.3f})')


## 7. (Optional) Interactive chat

Run the cell below for an interactive REPL (type `exit` to stop). Works in Colab because `input()` is supported.

In [ ]:
!python cli.py
